# ECG records: JSON/BIN batch summary (folder scan)

This notebook scans a folder for ECG record metadata JSON files with matching `.bin` files, computes per-record summaries using `summarize_record()` from `analyze_ecg_zip_structure_refactoring2.py`, collects results into a DataFrame, saves to CSV, and prints one row per record.

## What you need to change
- `ROOT_DIR`: folder that contains your records (can be the `recordings/` folder or a higher-level folder)
- `FS_HZ`: sampling frequency (Hz), e.g. 200
- `OUT_CSV`: output CSV file path
- `MODULE_PATH`: path to `analyze_ecg_zip_structure_refactoring2.py` (default: same folder as notebook)


In [ ]:
from __future__ import annotations

import importlib.util
import json, sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple


# --- User configuration ---
ROOT_DIR = Path("user-65955b5f50e02b125d4998ad/659ec124870b3d1d1630be39/recordings")      # <-- CHANGE ME
FS_HZ = 200                                   # <-- CHANGE ME
OUT_CSV = Path("ecg_records_summary.csv")

# Path to the helper module that contains summarize_record()
MODULE_PATH = Path("analyze_ecg_zip_structure_refactoring2.py")

if not MODULE_PATH.exists():
    raise FileNotFoundError(
        f"MODULE_PATH not found: {MODULE_PATH.resolve()}\n"
        "Place analyze_ecg_zip_structure_refactoring2.py next to this notebook, or set MODULE_PATH explicitly."
    )

# # Load the module from file (no install needed)
# spec = importlib.util.spec_from_file_location("ecg_analyze_mod", str(MODULE_PATH))
# assert spec and spec.loader
# ecgmod = importlib.util.module_from_spec(spec)
# spec.loader.exec_module(ecgmod)

# Load the module from file (no install needed)
spec = importlib.util.spec_from_file_location("ecg_analyze_mod", str(MODULE_PATH))
assert spec and spec.loader
ecgmod = importlib.util.module_from_spec(spec)
# IMPORTANT: register in sys.modules before exec_module (needed for dataclasses/type resolution)
sys.modules[spec.name] = ecgmod
spec.loader.exec_module(ecgmod)

print("Loaded module:", MODULE_PATH.resolve())
print("Root folder:", ROOT_DIR.resolve())
print("FS_HZ:", FS_HZ)


In [ ]:
def _infer_sequence_id(json_path: Path) -> str:
    """Best-effort: if path contains .../<sequence_id>/recordings/<id>.json, return <sequence_id>."""
    parts = list(json_path.parts)
    if "recordings" in parts:
        i = parts.index("recordings")
        if i > 0:
            return parts[i - 1]
    # fallback: use parent folder name
    return json_path.parent.name


def _find_record_json_bin_pairs(root_dir: Path) -> List[Tuple[str, str, Path, Path]]:
    """Return [(sequence_id, recording_id, json_path, bin_path), ...] for JSONs that have a sibling BIN."""
    if not root_dir.exists() or not root_dir.is_dir():
        raise NotADirectoryError(f"ROOT_DIR not found or not a directory: {root_dir}")

    ignore_names = {"gaps.json", "report.json"}
    json_files = [p for p in root_dir.rglob("*.json") if p.name not in ignore_names]

    pairs: List[Tuple[str, str, Path, Path]] = []
    for jp in sorted(json_files):
        bp = jp.with_suffix(".bin")
        if bp.exists():
            seq = _infer_sequence_id(jp)
            rid = jp.stem
            pairs.append((seq, rid, jp, bp))
    return pairs


def _load_json_with_fixes(json_path: Path) -> Dict[str, Any]:
    """Load JSON via the module helper and normalize field naming when needed."""
    data = ecgmod._safe_json_load_path(json_path)
    if not isinstance(data, dict):
        return {}

    # Many datasets use 'noises_annotated' instead of 'noises'.
    # The module's summarize_record() reads 'noises', so map it if needed.
    if "noises" not in data and "noises_annotated" in data:
        data["noises"] = data.get("noises_annotated")
    return data


def _guess_bytes_per_sample(bin_bytes: int, rpeaks_last_idx: Optional[int]) -> int:
    """Heuristic: decide between 2B (int16) and 4B (int32) using last rpeak index when available."""
    if rpeaks_last_idx is None or rpeaks_last_idx < 0:
        return 2
    target = rpeaks_last_idx + 1
    cand2 = bin_bytes // 2
    cand4 = bin_bytes // 4
    valid2 = cand2 >= target
    valid4 = cand4 >= target
    if valid2 and valid4:
        return 2 if (cand2 - target) <= (cand4 - target) else 4
    if valid4 and not valid2:
        return 4
    return 2


def _last_rpeak_sample_index(meta: Dict[str, Any]) -> Optional[int]:
    rpeaks = meta.get("rpeaks")
    if not isinstance(rpeaks, list) or not rpeaks:
        return None
    idxs = []
    for rp in rpeaks:
        if isinstance(rp, dict) and isinstance(rp.get("sampleIndex"), int):
            idxs.append(rp["sampleIndex"])
    return max(idxs) if idxs else None


In [ ]:
pairs = _find_record_json_bin_pairs(ROOT_DIR)
print(f"Found {len(pairs)} JSON+BIN pairs")
if pairs[:5]:
    print("First 5 pairs:")
    for seq, rid, jp, bp in pairs[:5]:
        print(" -", seq, rid, jp)


In [ ]:
rows: List[Dict[str, Any]] = []

for seq, rid, json_path, bin_path in pairs:
    # summarize_record() expects (sequence_id, recording_id, json_ref, bin_ref) + loader/size fns
    rec, rec_dict = ecgmod.summarize_record(
        sequence_id=seq,
        recording_id=rid,
        json_ref=json_path,
        bin_ref=bin_path,
        load_json_fn=_load_json_with_fixes,
        file_size_fn=lambda p: Path(p).stat().st_size,
        fs=FS_HZ,
    )

    # Optional: correct sample count/duration if bins are 4-byte samples (common in ZIVE files)
    try:
        meta = _load_json_with_fixes(json_path)
        last_idx = _last_rpeak_sample_index(meta)
    except Exception:
        meta = {}
        last_idx = None

    bin_bytes = int(rec_dict.get("bin_bytes", 0))
    bps = _guess_bytes_per_sample(bin_bytes, last_idx)
    samples_corr = (bin_bytes // bps) if bps in (2, 4) else None
    duration_corr = (samples_corr / FS_HZ) if (samples_corr is not None and FS_HZ > 0) else None

    nz_samples = int(rec_dict.get("_annotated_noises_samples", 0))
    noises_fraction_corr = (nz_samples / samples_corr) if (samples_corr and samples_corr > 0) else None

    row = dict(rec_dict)
    row["json_path"] = str(json_path)
    row["bin_path"] = str(bin_path)
    row["bytes_per_sample_guess"] = bps
    row["samples_corrected"] = int(samples_corr) if samples_corr is not None else None
    row["duration_s_corrected"] = float(duration_corr) if duration_corr is not None else None
    row["annotated_noises_fraction_corrected"] = float(noises_fraction_corr) if noises_fraction_corr is not None else None

    rows.append(row)

df = pd.DataFrame(rows)
print("Rows collected:", len(df))


In [ ]:
# Format columns similarly to the original script
if not df.empty:
    if "flags" in df.columns:
        df["flags"] = df["flags"].apply(lambda x: "|".join(x) if isinstance(x, list) else (x or ""))
    if "json_ok" in df.columns:
        df["json_ok"] = df["json_ok"].astype(bool)

    # Drop helper key from module
    if "_annotated_noises_samples" in df.columns:
        df = df.drop(columns=["_annotated_noises_samples"])

    # Reorder a useful subset first, keep the rest after
    preferred = [
        "sequence_id", "recording_id", "user_id", "channel_count",
        "bin_bytes", "bytes_per_sample_guess",
        "samples", "samples_corrected",
        "duration_s", "duration_s_corrected",
        "flags_count", "flags",
        "rpeaks_count", "ann_n_count", "ann_s_count", "ann_v_count", "ann_u_count",
        "annotated_noises_count", "annotated_noises_fraction", "annotated_noises_fraction_corrected",
        "has_comment", "json_ok", "json_keys_correct",
    ]
    cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
    df = df[cols]

# Write CSV
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_CSV, index=False)
print("Wrote:", OUT_CSV.resolve())


In [ ]:
# Print every record as a single row (header once)
if df.empty:
    print("No records found.")
else:
    # Adjust display so each record prints in one row without wrapping as much as possible
    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.width", 200,
        "display.max_colwidth", 120,
    ):
        print(df.to_string(index=False))
